# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainabFatima-hzf/ML-flyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Rule, in plain words:** a page is worth reviewing for a refresh if (a) it hasn't been touched
in a long time, **and** (b) it's ranking well enough to be seen but people aren't clicking it —
the classic "the page is visible, but something about it is failing to convert that visibility
into clicks" pattern. This directly mirrors two flag families from the session: staleness behind
the refresh flags, and CTR-vs-position behind the needs-CTR-fix logic.

**Two signals, checked first, before I trust them in a rule** — bucket tables with real `n` and
the ACTUAL forward-decline rate, using the honest label from Week 3
(`declined_next_28d`: prior 28 days vs. the 28 days after `decision_date`, `2026-03-15`, on the
March 2026 slice).

In [ ]:
# --- Self-contained setup (this notebook doesn't depend on w03 having run) ---
!pip install -q duckdb huggingface_hub

import duckdb, os
import pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql(f"""
    CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{os.environ["HF_TOKEN"]}')
""")

REPO        = "hf://datasets/FlyRank/internship-warehouse"
DAILY       = f"{REPO}/fact_content_daily_performance/*/*.parquet"
DIM_CONTENT = f"{REPO}/dim_content.parquet"

DECISION_DATE = "2026-03-15"
PRIOR_START   = "2026-02-15"   # 28 days strictly before decision_date
NEXT_END      = "2026-04-12"   # 28 days strictly after decision_date

# Prior-window features (same construction as w03 -- knowable before decision_date)
features = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_impressions) AS prior_28d_impressions,
        SUM(gsc_clicks)      AS prior_28d_clicks,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS prior_28d_avg_position
    FROM read_parquet('{DAILY}')
    WHERE report_date >= '{PRIOR_START}' AND report_date < '{DECISION_DATE}'
    GROUP BY content_hash_id, client_hash_id
""").df()

# Forward-window label (same honest proxy as w03 -- never fed to the rule as an input)
label_frame = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS next_28d_impressions
    FROM read_parquet('{DAILY}')
    WHERE report_date >= '{DECISION_DATE}' AND report_date < '{NEXT_END}'
    GROUP BY content_hash_id, client_hash_id
""").df()

# Content-level staleness signal: days since the page was last updated, AS OF decision_date
content_meta = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           DATE '{DECISION_DATE}' - content_updated_date AS days_since_update,
           DATE '{DECISION_DATE}' - content_created_date AS content_age_days
    FROM read_parquet('{DIM_CONTENT}')
    WHERE is_deleted IS NOT TRUE
""").df()

demo = (
    features
    .merge(label_frame, on=["content_hash_id", "client_hash_id"], how="inner")
    .merge(content_meta, on=["content_hash_id", "client_hash_id"], how="inner")
)
demo["declined_next_28d"] = (demo["next_28d_impressions"] < demo["prior_28d_impressions"]).astype(int)
demo["ctr"] = (demo["prior_28d_clicks"] / demo["prior_28d_impressions"].replace(0, pd.NA))

print("Rows ready for scoring:", len(demo))
print("Base rate (declined_next_28d):", round(demo["declined_next_28d"].mean(), 3))


**Signal A — staleness (behind the refresh flags):** bucket `days_since_update` and look at
the real forward-decline rate in each bucket, with `n` printed for every bucket.

In [ ]:
bins   = [-1, 90, 180, 365, 10_000]
labels = ["<90d", "90-180d", "180-365d", "365d+"]
demo["staleness_bucket"] = pd.cut(demo["days_since_update"].dt.days, bins=bins, labels=labels)

staleness_table = (
    demo.groupby("staleness_bucket", observed=True)
    .agg(n=("declined_next_28d", "size"), decline_rate=("declined_next_28d", "mean"))
    .round(3)
)
print(staleness_table)
print()
print("Overall base rate for reference:", round(demo['declined_next_28d'].mean(), 3))
print()
print("VERDICT (fill in after reading the table -- one word: CONFIRMED / OPPOSITE / MIXED / FALSE):")
print("  -> does decline_rate clearly RISE as staleness_bucket increases? That's the assumption")
print("     behind FlyRank's refresh flags -- 'stale' implies 'at risk'. Check it, don't assume it.")


**Signal B — CTR-vs-position (behind the needs-CTR-fix logic):** among pages that are visible
(decent impressions) and ranking well (position 1-20), does a LOW click-through rate predict
higher forward decline? Bucket CTR, restricted to that visible/ranked slice, with `n` per bucket.

In [ ]:
visible_ranked = demo[
    (demo["prior_28d_impressions"] >= 100) &
    (demo["prior_28d_avg_position"] > 0) &
    (demo["prior_28d_avg_position"] <= 20)
].copy()

ctr_bins   = [-0.01, 0.005, 0.02, 1.0]
ctr_labels = ["low_ctr(<0.5%)", "mid_ctr(0.5-2%)", "high_ctr(>2%)"]
visible_ranked["ctr_bucket"] = pd.cut(visible_ranked["ctr"], bins=ctr_bins, labels=ctr_labels)

ctr_table = (
    visible_ranked.groupby("ctr_bucket", observed=True)
    .agg(n=("declined_next_28d", "size"), decline_rate=("declined_next_28d", "mean"))
    .round(3)
)
print(ctr_table)
print()
print("Base rate within this visible/ranked slice:", round(visible_ranked['declined_next_28d'].mean(), 3))
print()
print("VERDICT (fill in after reading the table):")
print("  -> does decline_rate come out HIGHER in the low_ctr bucket than high_ctr? That's the")
print("     needs-CTR-fix assumption -- check the real numbers, a clean negative here is fine too.")


## 2. Build the ranked queue (writes the CSV)

**One rule, transparent and hand-written — no fitted weights.** Both gates from Section 1 have to
pass before a page even earns a nonzero score; the score itself is just prior-window impressions
(the "how much is at stake" multiplier), so bigger, more-visible pages that also trip both gates
rank first. One reason code describes the whole rule (this IS the rule, so it only ever needs one
label); one constant action label, since this rule only ever recommends one kind of action.

In [ ]:
stale_gate = (demo["days_since_update"].dt.days >= 180).astype(int)
ctr_gate = (
    (demo["prior_28d_impressions"] >= 100) &
    (demo["prior_28d_avg_position"] > 0) & (demo["prior_28d_avg_position"] <= 20) &
    (demo["ctr"] < 0.02)
).astype(int)

# Transparent score: readable on purpose, no fitted weights.
demo["score"] = stale_gate * ctr_gate * demo["prior_28d_impressions"]

demo["reason_code"] = "stale_visible_low_ctr_page"
demo["action_label"] = "review_for_refresh"

ranked = demo.sort_values("score", ascending=False).reset_index(drop=True)

n_scored = (ranked["score"] > 0).sum()
print(f"{n_scored} / {len(ranked)} pages clear both gates and get a nonzero score.")
print(f"Base rate among scored pages: {ranked.loc[ranked['score'] > 0, 'declined_next_28d'].mean():.3f}")
print(f"Overall base rate: {ranked['declined_next_28d'].mean():.3f}")

import os
os.makedirs("work/outputs", exist_ok=True)
out_cols = [
    "content_hash_id", "client_hash_id", "score", "reason_code", "action_label",
    "prior_28d_impressions", "prior_28d_clicks", "prior_28d_avg_position",
    "days_since_update", "declined_next_28d",
]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Written: work/outputs/baseline_action_score.csv")


In [ ]:
# precision@K -- the honest way to read a ranked queue (building-baselines skill)
def precision_at_k(labels_sorted_by_score, k):
    return labels_sorted_by_score[:k].mean()

for k in [50, 200, 500]:
    p_at_k = precision_at_k(ranked["declined_next_28d"].values, k)
    print(f"precision@{k}: {p_at_k:.3f}   (base rate: {ranked['declined_next_28d'].mean():.3f})")


## 3. Top-10 review

For each of the top 10 in the queue: the action, why it's there (in the rule's own terms), and
what would make this specific pick wrong. Generated straight from each row's real values, so this
reads honestly instead of me describing rows I never looked at.

In [ ]:
top10 = ranked.head(10).copy()

for i, row in top10.iterrows():
    days_stale = row["days_since_update"].days
    print(f"#{i+1} | content_hash_id={row['content_hash_id']}")
    print(f"  Action: {row['action_label']}   Reason code: {row['reason_code']}")
    print(f"  Why: {days_stale} days since last update (stale gate passed); "
          f"CTR {row['ctr']*100:.2f}% at avg position {row['prior_28d_avg_position']:.1f} "
          f"(low-CTR gate passed); {row['prior_28d_impressions']:.0f} prior impressions "
          f"driving the score.")
    print(f"  What would make this wrong: if this page was already refreshed after the decision "
          f"date but the update timestamp hasn't synced yet, or if the low CTR is seasonal/ "
          f"expected for this query type rather than a real problem, this pick is a false "
          f"positive -- worth a 5-second human glance before committing writer time.")
    print(f"  Actually happened (for my own read, not fed to the rule): "
          f"declined_next_28d = {row['declined_next_28d']}")
    print()


## 4. Weak picks + leakage check

**Weak-pick scan (automatic, then eyeball the printed top-10 above for a second opinion):** flags
any top-10 row riding on thin evidence — barely clearing a gate rather than clearly clearing it —
since those are exactly the kind of pick that looks fine in a table but falls apart on inspection.

In [ ]:
# Automatic red flags: rows that barely cleared a gate rather than clearing it with room to spare
top10["thin_staleness"] = top10["days_since_update"].dt.days.between(180, 200)
top10["thin_ctr"] = top10["ctr"].between(0.015, 0.02)

weak = top10[top10["thin_staleness"] | top10["thin_ctr"]]
print(f"{len(weak)} / 10 top picks are riding on a gate they only just barely cleared:")
print(weak[["content_hash_id", "days_since_update", "ctr", "score"]])
print()
print("-> even a single weak pick here is expected and fine (the building-baselines skill says")
print("   so directly): if the scan above found zero, look harder at the top 10 printed in")
print("   Section 3 by eye -- a rule this simple should not bat 10 for 10.")
print()

# --- Leakage check: confirm no product flags or future-window inputs anywhere in the rule ---
rule_inputs = ["days_since_update", "prior_28d_impressions", "prior_28d_avg_position", "ctr"]
future_window_cols = {"next_28d_impressions", "declined_next_28d"}
product_flag_cols = {"health_score", "priority_score", "action_type", "is_quick_win", "needs_ctr_fix"}

leak_check_1 = future_window_cols.intersection(rule_inputs)
leak_check_2 = product_flag_cols.intersection(rule_inputs)
print("Future-window columns used as rule inputs (should be empty set):", leak_check_1)
print("Product-decision columns used as rule inputs (should be empty set):", leak_check_2)
assert not leak_check_1 and not leak_check_2, "LEAKAGE: a future-window or product-flag column reached the rule!"
print("Confirmed clean: the rule only ever sees prior-window signals, and product flags")
print("were never in this dataset to begin with (per the flyrank-data / flyrank-context skills).")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.